# Kaggle finetune + ONNX export for SkyTNT/midi-model

End-to-end recipe for the diploma project. Runs on a single T4 GPU.

## Inputs you must attach to this notebook
1. **This repo** uploaded as a Kaggle Dataset (or `git clone`'d in cell 2). Folder name doesn't matter — we cd into it.
2. **Your raw MIDI** as a Kaggle Dataset (e.g. `your-username/my-midi-corpus`). Files can be nested.

## Outputs (under `/kaggle/working/artifacts/`)
- `onnx/model_base.onnx`
- `onnx/model_token.onnx`
- `tokenizer/tokenizer_config.json`
- `examples/sample_*.mid` (smoke test)

Download the artifacts at the end of the run, drop them under `artifacts/` in your local repo, and the JUCE plugin will pick them up automatically.

## 1. Environment setup
Enable GPU + Internet under "Settings" before running.

In [ ]:
import os, subprocess, sys

# Pick the dataset path Kaggle gave us.
REPO_INPUT = "/kaggle/input/midi-generation-plugin"   # your code dataset
MIDI_INPUT = "/kaggle/input/my-midi-corpus"           # your MIDI dataset

if not os.path.exists(REPO_INPUT):
    !git clone --depth 1 https://github.com/SkyTNT/midi-model.git /kaggle/working/midi-model
    raise RuntimeError("Attach the repo as a Kaggle dataset and set REPO_INPUT.")

WORKDIR = "/kaggle/working/repo"
!cp -r "$REPO_INPUT" $WORKDIR
%cd $WORKDIR
!ls

In [ ]:
!pip install -q -r requirements.txt

## 2. Sanity-check the dataset
Counts how many MIDI files we found.

In [ ]:
import os

n = 0
for root, _, files in os.walk(MIDI_INPUT):
    for f in files:
        if f.lower().endswith((".mid", ".midi")):
            n += 1
print(f"Found {n} MIDI files under {MIDI_INPUT}")
assert n > 0, "No MIDI files found - check MIDI_INPUT."

## 3. Download SkyTNT pretrained weights
(Skip if you have your own pretraining.)

In [ ]:
from huggingface_hub import snapshot_download

PRETRAINED_REPO = "skytnt/midi-model-tv2o-medium"
pretrained_dir = snapshot_download(repo_id=PRETRAINED_REPO, allow_patterns=["*.safetensors", "*.bin", "*.ckpt", "config.json"])
print("pretrained at:", pretrained_dir)
!ls -la "$pretrained_dir"

## 4. Finetune
T4 has 16 GB VRAM, so use bf16 mixed precision + small batch + grad accumulation.

In [ ]:
%env PYTHONUNBUFFERED=1
!python -m skytnt_adapter.finetune_skytnt \
    --data "$MIDI_INPUT" \
    --pretrained "$pretrained_dir" \
    --output checkpoints/skytnt \
    --config tv2o-medium \
    --max-len 2048 \
    --batch-size 1 --batch-size-val 1 \
    --workers 2 --workers-val 2 \
    --acc-grad 4 \
    --max-step 4000 --warmup-step 200 --val-step 400 \
    --lr 1e-4 \
    --precision bf16-mixed \
    --accelerator gpu --devices 1

## 5. Export to ONNX
Produces both `model_base.onnx` and `model_token.onnx` plus `tokenizer_config.json`.

In [ ]:
!python -m skytnt_adapter.export_skytnt_onnx \
    --ckpt checkpoints/skytnt \
    --config tv2o-medium \
    --out-dir /kaggle/working/artifacts/onnx \
    --tokenizer-out /kaggle/working/artifacts/tokenizer/tokenizer_config.json

## 6. Smoke test the exported ONNX models
Generates 2 short MIDI snippets to confirm the runtime works.

In [ ]:
!python -m skytnt_adapter.sample_generate \
    --mode onnx \
    --base /kaggle/working/artifacts/onnx/model_base.onnx \
    --token /kaggle/working/artifacts/onnx/model_token.onnx \
    --tokenizer /kaggle/working/artifacts/tokenizer/tokenizer_config.json \
    --out /kaggle/working/artifacts/examples \
    --num 2 --max-len 256

## 7. Bundle artifacts
After the cell below, download `/kaggle/working/artifacts.zip` from the Output panel and drop the contents into `artifacts/` in your local repo.
The JUCE plugin auto-discovers them at startup.

In [ ]:
!cd /kaggle/working && zip -r artifacts.zip artifacts && ls -la artifacts.zip